# Trader Metrics

No API keys needed in this, no trades are actioned, only an analysis of public market data.

1. Data sourced live using `CCXT` API library, no scraping required.
2. Indicators derived using `pandas-ta-classic`, replacing `numpy.2`



### Supertrend Integration

The triple-Supertrend signal from the live bot (`inputs/supertrend.py`) is the trend-vote analogue of this chapter's four-vote ranking: three ATR-channel bands (periods 12/3, 10/1, 11/2) must agree on an uptrend, gated by EMA-200. Where Chapter One ranks coins by Buy / EMA-vs-Kalman / AMAT / RSI votes, the Supertrend adds a trend vote whose agreement count and reversal flip became first-class model inputs downstream.

How and where it was integrated:
- **Features** -> `inputs/build_dataset_1h.py` `supertrend_block()` emits the `f_st_` family (band agreement, signed band distances, reversal flip, EMA-200 distance), consumed by the Chapter Three model. Causal and scale-invariant, like every other `f_` feature.
- **Signal / benchmark** -> the bot's entry/exit rules are scored after fees in `inputs/baseline_supertrend_1h.py` (Chapter Three, Stability).
- **Full record** -> `tasks/integration-2026-06-23-claudetrader-supertrend.md`.

## Design Update

Chapter One is the **metrics layer** - the indicator stack and the Four Vote scoring that rank
coins on the live daily scan. The **model track** has since moved to a 1-hour, full-market frame
(every active USDT spot pair, point-in-time screened; see the README "Data and Model Design" and
`tasks/data-standards.md`). The same indicator ideas carry over there in two window families - a
wall-clock family equal to these daily windows times 24, and a shorter intraday family - alongside
optional pandas-ta and TA-Lib breadth, all built by `inputs/build_dataset_1h.py`. This notebook
remains the daily-scan view of the signal stack; the 1h model is trained and scored in Chapter Three.

### Environment Setup

Dependencies are pinned in `inputs/requirements.txt` and reinstalled on each fresh kernel,`venv` also committed. The core libraries are `ccxt` (data), `pandas-ta` (indicators), `pykalman` (smoothing), and `plotly` (charts). Run the setup cell below once per session to rebuild the environment.

Make sure to log into Binance and Alpaca accounts before running this setup below. To avoid api issues, the preferred Binance.US login is accessed here: https://docs.ccxt.com/en/latest/exchange-markets.html

In [12]:
import sys, subprocess
from pathlib import Path

# assign Python kernel,
REQUIRED_PY = (3, 11)
if sys.version_info[:2] != REQUIRED_PY: raise RuntimeError(
        f"This notebook needs Python {REQUIRED_PY[0]}.{REQUIRED_PY[1]}, "
        f"but the kernel is {sys.version.split()[0]}. Switch the kernel and re-run.")

# Install packages & versions in requirements.txt
REQUIREMENTS = Path("../inputs/requirements.txt")
if not REQUIREMENTS.exists():
    raise FileNotFoundError(f"{REQUIREMENTS} not found — run from the repo root.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
     "--break-system-packages", "--disable-pip-version-check",
     "-r", str(REQUIREMENTS)], check=True,)

# import package modules and naming
import warnings, numpy as np, pandas as pd
from datetime import datetime
import ccxt
import pandas_ta_classic as ta
from pykalman import KalmanFilter
import plotly, plotly.graph_objects as go, plotly.io as pio
warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype+notebook_connected"  

# quick sanity check
print(f"Environment ready  ·  python {sys.version.split()[0]}  ({sys.executable})")
print(f"  ccxt {ccxt.__version__} · pandas {pd.__version__} · "
      f"numpy {np.__version__} · plotly {plotly.__version__}")

Environment ready  ·  python 3.11.13  (/Users/seamus/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/bin/python)
  ccxt 4.5.59 · pandas 2.3.3 · numpy 2.4.6 · plotly 6.8.0


### Data Sources and Provenance

This chapter scans the market LIVE through ccxt: it pulls the recent and still-forming bars straight
from the exchange to rank tonight's buy candidates. That live feed is the right tool for a decision at
the open, but it is not what the model is trained or backtested on.

The training and backtest frames are separate, offline, and survivorship-complete. They are built in
chapter 3 from the official Binance public archive at `data.binance.vision`: monthly per-symbol zips of
OHLCV klines, one file per interval, exchange-direct and checksummed so the build is deterministic and
reproducible. ccxt is only a live top-up for the bar still forming, never the historical record.

Three decision frames are pulled at three resolutions, each its own dataset:

| frame | raw klines | built dataset | role |
| --- | --- | --- | --- |
| 1h (existing) | `klines_1h/` | `dataset_1h_allmarket.parquet` | original day-trade frame |
| 4h (new, 2026-06-23) | `klines_4h/` | `dataset_4h_allmarket.parquet` | coarser frame where after-fee edge may survive |
| daily (planned) | `klines/` | next in the plan | trend / context frame |

The same Supertrend and indicator definitions used in the scan below apply to the historical frames, so
a candidate that scans well live can be checked against history. The cell below proves the source: it
pulls one month of bars at 1h and 4h directly from the archive.

In [13]:
# Demo: how the OLD (1h, daily) and NEW (4h) data were downloaded, and from where. Every frame
# sources the SAME official Binance public archive at data.binance.vision -- monthly per-symbol
# OHLCV kline zips, one file per interval, exchange-direct and checksummed (NOT a live API). ccxt
# is only a live top-up for the bar still forming. This cell pulls one month at 1h and 4h to prove
# the source and format; the full pull is the commands in the table.
import io, zipfile, urllib.request, pandas as pd
BINANCE_VISION = "https://data.binance.vision/"
def vision_url(symbol, interval, month):
    return f"{BINANCE_VISION}data/spot/monthly/klines/{symbol}/{interval}/{symbol}-{interval}-{month}.zip"

provenance = pd.DataFrame([
    ("1h  (old)", "acquire_vision.py download --interval 1h", "klines_1h/<SYM>/", "dataset_1h_allmarket.parquet"),
    ("1d  (old)", "flow_data.py --interval 1d --all-market",  "klines/<SYM>/",    "daily_flow.csv"),
    ("4h  (new)", "acquire_vision.py download --interval 4h", "klines_4h/<SYM>/", "dataset_4h_allmarket.parquet"),
], columns=["frame", "download command", "raw klines on disk", "built dataset"])
print("Survivorship-complete: acquire_vision.py crawl enumerates EVERY symbol ever listed (incl.")
print("delisted), so dead coins are not silently dropped the way the exchangeInfo path is.\n")
print(provenance.to_string(index=False))

print("\nLIVE proof of source + format (one month of BTCUSDT, straight from the archive):")
for interval in ("1h", "4h"):
    u = vision_url("BTCUSDT", interval, "2024-01")
    try:
        raw = urllib.request.urlopen(u, timeout=15).read()
        z = zipfile.ZipFile(io.BytesIO(raw))
        bars = pd.read_csv(io.BytesIO(z.read(z.namelist()[0])), header=None).iloc[:, :6]
        bars.columns = ["open_time", "open", "high", "low", "close", "volume"]
        print(f"  {interval}: {len(bars):3d} bars, {len(raw)/1024:4.0f} KB  <-  {u}")
    except Exception as e:
        print(f"  {interval}: offline ({e}); full archive lives under inputs/binance-data/klines_{interval}/")


Survivorship-complete: acquire_vision.py crawl enumerates EVERY symbol ever listed (incl.
delisted), so dead coins are not silently dropped the way the exchangeInfo path is.

    frame                         download command raw klines on disk                built dataset
1h  (old) acquire_vision.py download --interval 1h   klines_1h/<SYM>/ dataset_1h_allmarket.parquet
1d  (old)  flow_data.py --interval 1d --all-market      klines/<SYM>/               daily_flow.csv
4h  (new) acquire_vision.py download --interval 4h   klines_4h/<SYM>/ dataset_4h_allmarket.parquet

LIVE proof of source + format (one month of BTCUSDT, straight from the archive):
  1h: 744 bars,   42 KB  <-  https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2024-01.zip
  4h: 186 bars,   11 KB  <-  https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/4h/BTCUSDT-4h-2024-01.zip


### Import Data

Live price data is sourced between 24 hours and 1 week from multiple trading platforms. Five filters were highlighted below, including `EXCHANGE`, `SANDBOX`, `TIMEFRAME`, `LIMIT`, and `TOP_N`. Currently, the`ccxt` library of APIs maintains access to 105 leading crypto [exchange platforms](https://github.com/ccxt/ccxt/wiki/Exchange-Markets). Happy browsing!

In [14]:
EXCHANGE  = "binance"   # 105 platforms available, see https://github.com/ccxt/ccxt/wiki/Exchange-Markets)
SANDBOX   = False       # True = testnet / fake money (mainly for the trading step later)
TIMEFRAME = "1d"        # candle size: "1m","5m","1h","4h","1d","1w"
LIMIT     = 300         # candles pulled per coin (max 1000)
TOP_N     = 20          # how many of the most-traded /USDT pairs to scan

exchange = getattr(ccxt, EXCHANGE)()
exchange.set_sandbox_mode(SANDBOX)
print(f"Configured: {EXCHANGE} | sandbox={SANDBOX} | {TIMEFRAME} candles x{LIMIT} | top {TOP_N}")

Configured: binance | sandbox=False | 1d candles x300 | top 20


In [15]:
def calculate_indicator(symbol, timeframe=TIMEFRAME, limit=LIMIT):
    bars = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(bars[:-1], columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.set_index("timestamp")
    close, low = df["close"].iloc[-1], df["low"].iloc[-1]

    # Kalman threshol & smoother
    kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1],
                      initial_state_mean=0, initial_state_covariance=1,
                      observation_covariance=1, transition_covariance=.01)
    state_means, _ = kf.filter(df["close"].values)
    df["kf_mean"] = state_means
    kalman = df["kf_mean"].iloc[-1]
    above_kalman = bool(low > kalman)

    # Trend: Is EMA-14 leading the Kalman mean?
    df.ta.ema(length=14, append=True)
    ema_cross = bool(df["EMA_14"].iloc[-1] > kalman)

    # Bollinger(14) mean-reversion envelope.
    bb = df.ta.bbands(length=14)
    bbl, bbu = bb["BBL_14_2.0"].iloc[-1], bb["BBU_14_2.0"].iloc[-1]

    # Ichimoku projectionse.
    ich = df.ta.ichimoku()[1]
    isa_9, isb_26 = ich["ISA_9"].iloc[-1], ich["ISB_26"].iloc[-1]

    # Archer MA trend flag, RSI, Choppiness.
    amat = bool(df.ta.amat()["AMATe_LR_8_21_2"].iloc[-1] == 1)
    rsi = float(df.ta.rsi().iloc[-1])
    chop = round(float(df.ta.chop().iloc[-1]), 2)
    
    # Candle shape check: any doji / dragonfly / gravestone candles?
    o, h, l, c = df["open"].iloc[-1], df["high"].iloc[-1], df["low"].iloc[-1], df["close"].iloc[-1]
    rng   = h - l
    body  = abs(c - o)
    upper = h - max(o, c)
    lower = min(o, c) - l
    doji       = bool(rng > 0 and body <= 0.10 * rng)
    dragonfly  = bool(doji and lower >= 0.6 * rng and upper <= 0.10 * rng)
    gravestone = bool(doji and upper >= 0.6 * rng and lower <= 0.10 * rng)

    # MACD (12,26,9)
    macd_line   = df["close"].ewm(span=12, adjust=False).mean() - df["close"].ewm(span=26, adjust=False).mean()
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    macd, sig           = macd_line.iloc[-1], signal_line.iloc[-1]
    macd_prev, sig_prev = macd_line.iloc[-2], signal_line.iloc[-2]
    macd_buy  = bool(macd_prev <= sig_prev and macd > sig)
    macd_sell = bool(macd_prev >= sig_prev and macd < sig)
    macd_below_zero = bool(macd < 0)
    buy  = amat and ema_cross and above_kalman
    sell = (not amat) and (not ema_cross) and (not above_kalman)    
    row = dict(Symbol=symbol, Buy=buy, Sell=sell, Close=round(float(close), 4),
               RSI=round(rsi, 2), Chop=chop, AMAT=amat,
               Ichimoku_9=round(float(isa_9), 4), Ichimoku_26=round(float(isb_26), 4),
               EMA_gt_Kalman=ema_cross, Low_gt_Kalman=above_kalman,
               Doji=doji, Dragonfly=dragonfly, Gravestone=gravestone,
               MACD=round(float(macd),4), MACD_Signal=round(float(sig),4),
               MACD_Buy=macd_buy, MACD_Sell=macd_sell, MACD_below_zero=macd_below_zero)
               
    return df, row

def plot(symbol):
    df, _ = calculate_indicator(symbol)
    fig = go.Figure(go.Candlestick(x=df.index, open=df.open, high=df.high, low=df.low, close=df.close, name=symbol))
    fig.add_trace(go.Scatter(x=df.index, y=df["kf_mean"], name="Kalman", line=dict(color="orange", width=2), opacity=0.7))
    fig.add_trace(go.Scatter(x=df.index, y=df["EMA_14"], name="EMA-14", line=dict(color="purple", width=2), opacity=0.7))
    fig.update_layout(title=symbol, xaxis_rangeslider_visible=False)
    return fig

def color_boolean(val):
    if val is True:  return "background-color: lightgreen"
    if val is False: return "background-color: pink"
    return "background-color: lightblue"

print("Engine ready.")

Engine ready.


### Analyze Data

The analysis pulls 300 daily candles and computes the following:

- Kalman Filter: a price smoothing noisy fluctuations, key indicator of close of day price strength
- Bollinger Bands: An elastic envelope derived from 14-day price average. Narrowing suggests quiet market, flaring suggest volatile.
- Ichimoku Spans: Two mean price projections used as trend map, suggesting uptrend or downtrends between 9 adn 26 days.
- AMAT: Archer Moving Average Trends used as binary indicators showing diverging fast and slow averages, or aligning showing sustained trend (1 = "trending up").
- Relative Strength Index (RSI). Speedometer, above 70, price running fast risks overselling and then cooling off. Below 30 may mean due to bounce.
- Choppiness Index: High values suggest inertia, low suggest clean trend.

Key values in these metrics in sell and buy point variables above, copied here again for review: `ema_crossover = ema_14 > ema_91` 

The originals scanned every Binance.US ticker, which is slow and noisy. Here we take the most liquid spot **/USDT** pairs by quote volume. Raise `TOP_N` to widen the scan; each extra symbol adds roughly a second.

In [16]:
tickers = exchange.fetch_tickers()
pairs = [(s, d.get('quoteVolume') or 0) for s, d in tickers.items()
         if s.endswith('/USDT') and ':' not in s]
universe = [s for s, _ in sorted(pairs, key=lambda x: -x[1])[:TOP_N]]
today = datetime.now().strftime('%Y-%m-%d')
print(f"{today}: scanning {len(universe)} pairs")
universe

2026-06-24: scanning 20 pairs


['BTC/USDT',
 'USDC/USDT',
 'ETH/USDT',
 'MEGA/USDT',
 'SOL/USDT',
 'USD1/USDT',
 'XRP/USDT',
 'ZEC/USDT',
 'BNB/USDT',
 'WLD/USDT',
 'DOGE/USDT',
 'TRX/USDT',
 'ADA/USDT',
 'XPL/USDT',
 'SPCXB/USDT',
 'EUR/USDT',
 'XAUT/USDT',
 'SUI/USDT',
 'NEAR/USDT',
 'PAXG/USDT']

### Rank Data
Run the engine over the universe into one table. The `try/except` skips symbols too new to have full indicator history. For example a token listed days ago has no Ichimoku cloud and skips these values instead of hiding them. 

In [17]:
rows = []
for symbol in universe:
    try: rows.append(calculate_indicator(symbol)[1])
    except Exception as e: print(f"skip {symbol}: {type(e).__name__}")
results = pd.DataFrame(rows)
print(f"\nscored {len(results)} pairs | {int(results.Buy.sum())} buys | {int(results.Sell.sum())} sells")
results.head()

skip SPCXB/USDT: KeyError

scored 19 pairs | 1 buys | 14 sells


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,BTC/USDT,False,True,62734.5700,37.36,53.33,False,65718.7900,70990.4550,False,False,False,False,False,-1977.5282,-2367.4442,False,False,True
1,USDC/USDT,True,False,1.0010,57.50,14.46,True,1.0033,1.0109,True,True,False,False,False,0.0001,0.0001,False,False,False
2,ETH/USDT,False,True,1667.1300,37.06,52.54,False,1759.3650,1964.7100,False,False,False,False,False,-65.5264,-81.5045,False,False,True
3,MEGA/USDT,False,True,0.0556,40.95,51.16,False,0.0590,0.0902,False,False,True,False,False,-0.0046,-0.0069,False,False,True
4,SOL/USDT,False,True,69.7100,43.01,50.31,False,71.8900,79.2700,False,False,False,False,False,-1.7340,-2.5827,False,False,True


In [18]:
top = (results.sort_values(['Buy','EMA_gt_Kalman','AMAT','RSI'], ascending=[False,False,False,True]).reset_index(drop=True).head(10))
bottom = (results.sort_values(['Sell','AMAT','RSI'],ascending=[False,False,False]).reset_index(drop=True).head(10))
print("ranked.")

if top['Buy'].any():
    print("Entry candidates showing bullish trends and Kalman lead prices:")
    for s in top.loc[top['Buy'], 'Symbol']: print(" ", s)
else: print("No long signals showing strongest-ranked names.")

top.style.map(color_boolean)

ranked.
Entry candidates showing bullish trends and Kalman lead prices:
  USDC/USDT


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,USDC/USDT,True,False,1.001000,57.500000,14.460000,True,1.003300,1.010900,True,True,False,False,False,0.000100,0.000100,False,False,False
1,XPL/USDT,False,False,0.088800,50.470000,41.670000,False,0.097300,0.091700,True,False,False,False,False,0.001800,0.001100,False,False,False
2,USD1/USDT,False,False,1.000400,50.790000,55.080000,False,1.000400,1.000100,True,False,True,False,False,0.000200,0.000200,False,False,False
3,WLD/USDT,False,False,0.543900,53.380000,47.830000,False,0.561600,0.474800,True,False,False,False,False,0.061400,0.067300,False,True,False
4,TRX/USDT,False,False,0.329300,49.490000,42.430000,True,0.328600,0.344200,False,True,False,False,False,-0.003300,-0.005500,False,False,True
5,EUR/USDT,False,True,1.139400,26.950000,42.470000,False,1.152700,1.159100,False,False,False,False,False,-0.004800,-0.003600,False,False,True
6,DOGE/USDT,False,True,0.078900,27.380000,47.040000,False,0.087200,0.098100,False,False,False,False,False,-0.004000,-0.003900,False,True,True
7,ADA/USDT,False,True,0.151500,28.190000,44.440000,False,0.181800,0.218700,False,False,False,False,False,-0.016000,-0.017000,False,False,True
8,XAUT/USDT,False,True,4090.840000,34.080000,50.110000,False,4257.130000,4389.715000,False,False,False,False,False,-83.331900,-80.854400,False,True,True
9,PAXG/USDT,False,True,4096.150000,34.100000,50.390000,False,4266.015000,4394.490000,False,False,False,False,False,-84.324000,-81.369700,False,True,True


A chart showing strongest buy candidate, with the Kalman and EMA-14 trend lines:

In [19]:
from IPython.display import HTML
HTML(plot(top['Symbol'].iloc[0]).to_html(include_plotlyjs="cdn", full_html=False))

In [20]:
# Save the chart as a standalone, self-contained HTML file you can open in any browser.
# include_plotlyjs=True embeds the library so it works offline. No nbformat / .show() needed.
fig = plot(top['Symbol'].iloc[0])
fig.write_html('../outputs/HTML/chart.html', include_plotlyjs=True, auto_open=False)
print('saved outputs/HTML/chart.html')


saved outputs/HTML/chart.html


## Performance Metrics

### MACD Trends

The metric of Moving Average Convergence Divergence is monitored as an indicator of changing momentum, which is quantified by the velocity of acceleration or decay in price trends (Appel, 1985). Because MACD is quantitatively faster than the units of EMA, these lines are expected to cross at the same moment the 12-EMA crosses the 26-EMA line. The nature of these crossovers are focus of day trading with buy points selected when MACD crosses above zero line, while sell points selected as it crosses below. Goes without saying, crossing above the zero line signals bullish trend, while passing below zero signals bearish trend. Tricky thing is while its earlier signals offer higher potential than the zero line, it carries risk of false starts.

Beneath the line graph, the histogram can add to our predictions providing distances between lines that guide us to when lines cross. The height of these distances represents momentum, so that a growing histogram indicates accelerating divergence. Alternatively, a shrinking histogram of converging lines suggests momentum is fading signalling a likely turn.

Divergence presents important metric in day trading that is least mechanical to read. Observing changing swings between MACD highs and lows, we can identify a shift in trends is expected. Bearish divergence can be characterised by higher highs while MACD show lower highs that caution an uptrend is hollow. Alternatively, a bullish divergence can appear with lower lows while MACD presents higher lows that caution a downtrend is exhausting. Need guidance on this, but seems divergence is mostly used to protect a position before change, rather than a precise entry to trade.

### Divergence Quadrants

![divergence quadrants](../outputs/PNG/divergence_matrix.png)

The full taxonomy compares the last two price swings against the matching MACD swings. Notation: HH higher high, HL higher low, LH lower high, LL lower low.

| Price | MACD | Source term | Standard term | Reading | Action |
|---|---|---|---|---|---|
| Higher high | Lower high | Divergence | Regular bearish | reversal / retracement | SELL |
| Lower high | Higher high | Convergence | Hidden bearish | trend continuation (down) | SELL |
| Higher low | Lower low | Divergence | Hidden bullish | trend continuation (up) | BUY |
| Lower low | Higher low | Convergence | Regular bullish | reversal / retracement | BUY |

Two cautions raised in the research. First, terminology is not standardised: GoodCrypto defines "divergence" as price stronger than the indicator and "convergence" as price weaker, which is the geometry above but the opposite word to how many texts use them. Read the HH/HL/LH/LL pattern, not the label. Second, a single chart can show mixed signals: in their BTC example the peaks diverged while the troughs converged, netting out bearish only because the peak structure dominated. Weigh both swing series, do not act on one line in isolation.

### Crypto MACD

MACD is well respected in BTC and ETH markets but cannot be handled in isolation. It is recommended to consider alongside position sizing and a expected stop. Some suggest a good crypto technique should draw the trendline on the MACD itself, not only on price. For example, in the May 2021 BTC top, price held a rising trendline while the MACD was already tracing a falling one. The MACD trendline broke first. The same setup appeared on the SOL 4h chart, pushing higher highs into resistance while MACD showed clearly lower highs. Was this a textbook bearish divergence, as it preceded the stall?

Timeline is important but does not lessen the MACD metric; a 4h or 1h MACD updates faster than a daily one. The real limiter to watch out for is volatility: the more violent the asset the riskier its forecast, hence reliance on corroboration or triangulation in fast markets.

### Corroborating Metrics

MACD gives momentum but poor at signaling overbought/oversold thresholds, so traders may confirm crossover against a second indicator and only act when both agree. In rough order of their selective applications:

| Partner | What it adds | Entry rule | Exit |
|---|---|---|---|
| Relative Vigor Index | closing strength vs range | both cross same direction | MACD opposite cross |
| Money Flow Index | price + volume, few signals | MFI overbought/oversold then MACD cross | MACD opposite cross |
| TEMA (50) | triple-smoothed trend | price breaks TEMA and MACD crosses | contrary signal from both |
| TRIX | momentum oscillator | MACD cross matched by TRIX zero-cross | MACD cross, or looser, TRIX zero-cross |
| Awesome Oscillator | 5/34 SMA momentum | MACD cross confirmed by AO | both turn contrary |
| MA (20) | trend validation | price tests the 20-MA, then MACD crosses up | — |

The shared idea is the guardrail we already build into the metric: do not take a bare crossover, require corroboration. A second indicator is one form of that; our epsilon noise band and confirmation bars are another, internal to MACD itself. MFI's volume gate is the most distinct addition and the strongest candidate to add next, since the histogram and slope already cover much of what RVI, TRIX, and AO contribute.

### Final Metrics

The current iteration targets these metrics by defining the following variables, including `cross_up` and `cross_down` crossover points; as well as `guarded_buy` and `guarded_sell` constraints. These were applied as conservative measures around noise which we found recommended in majority of readings. Histogram variables were derived for both `hist_slope` and `converging` flags to highlight early fade nearing zero. Divergence points were constructed at swing pivot points for variables of `bear_div` and `bull_div` representing bearish and bullish quadrant data of regular bounds. MACD breaks and hidden divergence, which are not yet coded, are next target. `strength` is compiled from band clearances, slope spikes, zero field, and corroboration by divergence, which was numerically viewed as significantly weighted swing series.

![BTC preview](../outputs/PNG/preview_btc.png)

### Entry Points

Buy trends are displayed first with Kalman flags displayed and lowest RSI scores at the top. Sell points are presented as the mirror to these, which is also characteristic of how the computational checklist was built between them.

### Exit Points

In [21]:
if bottom['Sell'].any():
    print("Exit candidates trending down showing prices below Kalman:")
    for s in bottom.loc[bottom['Sell'], 'Symbol']: print(" ", s)
else: print("No short signals today; showing weakest-ranked names.")
bottom.style.map(color_boolean)

Exit candidates trending down showing prices below Kalman:
  NEAR/USDT
  SOL/USDT
  ZEC/USDT
  MEGA/USDT
  BNB/USDT
  XRP/USDT
  BTC/USDT
  ETH/USDT
  SUI/USDT
  PAXG/USDT


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,NEAR/USDT,False,True,1.982000,44.450000,55.120000,False,2.356000,2.166000,False,False,False,False,False,0.004600,0.038200,False,False,False
1,SOL/USDT,False,True,69.710000,43.010000,50.310000,False,71.890000,79.270000,False,False,False,False,False,-1.734000,-2.582700,False,False,True
2,ZEC/USDT,False,True,416.080000,42.540000,55.210000,False,462.487500,470.060000,False,False,False,False,False,-17.442200,-16.197400,False,True,True
3,MEGA/USDT,False,True,0.055600,40.950000,51.160000,False,0.059000,0.090200,False,False,True,False,False,-0.004600,-0.006900,False,False,True
4,BNB/USDT,False,True,578.080000,39.170000,51.290000,False,626.417500,651.100000,False,False,False,False,False,-13.789100,-12.994300,False,False,True
5,XRP/USDT,False,True,1.110300,37.440000,45.630000,False,1.200400,1.299900,False,False,False,False,False,-0.039300,-0.041800,False,False,True
6,BTC/USDT,False,True,62734.570000,37.360000,53.330000,False,65718.790000,70990.455000,False,False,False,False,False,-1977.528200,-2367.444200,False,False,True
7,ETH/USDT,False,True,1667.130000,37.060000,52.540000,False,1759.365000,1964.710000,False,False,False,False,False,-65.526400,-81.504500,False,False,True
8,SUI/USDT,False,True,0.704200,35.130000,51.440000,False,0.775500,1.042000,False,False,False,False,False,-0.051600,-0.054700,False,False,True
9,PAXG/USDT,False,True,4096.150000,34.100000,50.390000,False,4266.015000,4394.490000,False,False,False,False,False,-84.324000,-81.369700,False,True,True


### Exit Geometry and Entry Trend Context

The exit-geometry view (`inputs/exit_geometry_viz.py`), shared with chapter three, rendered here so the
exit points and the trend data around each entry sit beside this chapter's exit-point and ranking tables. It draws the three
Supertrend trailing lines and EMA-200, shades the trend window before each entry, and annotates the entry
with its Supertrend agreement, EMA-200 side, and RSI; beneath the price it tracks the Supertrend
agreement (0-3) and RSI so the trend building into each entry is legible. Read-only, and it shares the
live bot's exit mechanics.

In [23]:
# Exit geometry + entry trend context (single source: inputs/exit_geometry_viz.py)
import os, sys
for _up in (".", "..", "../.."):
    _cand = os.path.abspath(os.path.join(_up, "inputs"))
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand); break
import exit_geometry_viz as egv
egv.render("BTC/USDT", frame=4, show=True, save=False)   # display inline; chapter three saves the PNGs

ModuleNotFoundError: No module named 'matplotlib'

### Four Votes

The current model is fitted with four overlapping signal variables. In summary, each price point is assigned multiple votes according to the following scoring. An additional +1 vote is granted towards favorable buy points, while -1 votes are subtracted from majoring sell points.

MACD tracks momentum. It acts only on "guarded" crossovers that pass an extra safety check, and holds its stance: a guarded buy stays +1 until a guarded sell flips it to -1.

Moving-average crossover compares a fast price average (20 periods) with a slow one (50). Fast above slow leads to a vote of +1; fast below leads to a vote of -1.

Fibonacci is treated context-dependent. For example, we proposed a "golden pocket" of 0.5-0.618 retracement band where moves are often paused. In such an uptrend, a dip into that band leads to a vote of +1, suggesting to buy the dip; while in a downtrend, a rally into the band leads to a vote of -1, suggesting to sell the bounce; otherwise = 0.

Candles are designed to look for an "engulfing" bar, where one candle fully swallows the prior one, which was assumed to indicate a common reversal sign. Bullish trends lead to a vote of +1, bearish produces a vote of -1, and the vote only lasts a few bars before expiring.

We note some caution here. During our first iteration earlier today (06/18/2026), we detected a bug in the code deriving candles. A column renamed during its position change, which swapped open for close prices. This produced a pattern informed by invalid data. The data and results we are looking at here in this report were corrected since, which strictly follow the standard engulfing definition derived from valid data columns.

The four votes are summed into a weighted score, and a trade fires when the score first crosses the threshold:

```
score = w_macd*MACD + w_ma*MA + w_fib*Fib + w_candle*Candle      (weights default 1)
BUY  fires when score first reaches +threshold   (default +2)
SELL fires when score first reaches -threshold   (default -2)
```

Two-point agreement is the standard rule, and the stricter three-point version is used when a setup has fewer trades to learn from. The score is a net total rather than a head-count: a +2 can be two buy votes and no sells, or three buys against one sell.

### Sanity Check

Hourly data, the ten-coin universe, about 1000 bars (~41 days), 0.1% fee per side, long-flat, threshold 2. This window was a broad crypto downtrend.

| coin | strategy | buy & hold | trades | win % | max DD |
|---|---|---|---|---|---|
| BTC | -1.4% | -21.3% | 4 | 50 | -7.3% |
| ETH | -9.2% | -26.1% | 5 | 20 | -16.1% |
| SOL | -19.8% | -22.1% | 6 | 33 | -36.9% |
| BNB | -0.9% | -9.9% | 5 | 40 | -6.7% |
| XRP | -6.3% | -17.9% | 4 | 50 | -15.5% |
| ADA | -16.9% | -38.2% | 5 | 20 | -22.0% |
| AVAX | -8.0% | -33.6% | 6 | 33 | -16.7% |
| LINK | -19.7% | -20.3% | 7 | 29 | -25.4% |
| LTC | -10.1% | -23.4% | 4 | 25 | -17.1% |
| DOGE | -9.2% | -22.6% | 3 | 33 | -15.1% |
| mean | -10.1% | -23.5% | | | |

Read it plainly. The engine lost money on every coin, but lost roughly half of what holding lost, because staying flat through the downtrend avoided the worst of it. That is a drawdown-reduction property, not an edge. Beating buy-and-hold by being absent during a fall is easy and does not survive into a rising or sideways market, where sitting flat means missing gains. Win rates of 20 to 50 percent confirm there is no demonstrated predictive skill here yet.

Two notes on method, true of every backtest here. It only ever used information that would have been available at the time, nothing from hindsight. And a signal fires the moment the score crosses the threshold, so the dashboard shows more markers than the backtest actually trades; if a second buy signal fires while we already hold a position, it doesn't open a new one.

In chapter three, we test across regimes, not just in this falling one, so a bull and a sideways stretch are included. We also add a stop and a take-profit, since a flat-long rule with no risk control flatters drawdown. Only if the out-of-sample, multi-regime, after-fees result clearly beats both buy-and-hold and a coin-flip is any of this worth real money, and even then the operator owns the decision and the live switch stays off.

In [ ]:
stamp = datetime.now().strftime('%Y%m%d')
path = f'../outputs/CSV/DailySignals_{stamp}.csv'
results.to_csv(path, index=False)
print("saved", path)

saved ../outputs/CSV/DailySignals_20260621.csv


### Decision Checklist

1. Read balances (USDT, BTC, ...).
2. Size each trade as cash divided by the number of long candidates.
3. Act on exits (`bottom`) before entries (`top`).
4. Widen timelines and trend comparisons (all data saved & dated in `/outputs/`).

##### Report Formats

 - `quarto preview day-metrics.ipynb --no-execute`
 - `quarto render day-metrics.ipynb --to html --no-execute`
 - `quarto render day-metrics.ipynb --to docx --no-execute`
 - `quarto render day-metrics.ipynb --to pdf --no-execute`